# RGCA Research-Safe Kaggle Baseline

This notebook assumes the dataset has **not** been set up yet.

It separates dataset preparation from experiment execution and prevents mock/debug runs from being mistaken for real research evidence.

Execution modes:

- `debug`: code smoke tests only; demo data allowed.
- `stress`: real MIMIC subset required; controlled retrieval-copy stress tests allowed.
- `real`: real MIMIC subset + local images + real retriever/generator required. This currently fails by design until real VLM/image retrieval backends are implemented.

No GCloud is used.


## 1. Repository Setup


In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_ROOT = Path('/kaggle/working/RGCA')
if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/pidoxy/RGCA.git', str(PROJECT_ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'pull', 'origin', 'main'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT_ROOT)], check=True)
print('Repo ready:', PROJECT_ROOT)


## 2. Choose Mode And Dataset Setup

For the current baseline, use `EXECUTION_MODE = "stress"`.

Set your PhysioNet username. The preparation cell will ask for your password securely.


In [ ]:
EXECUTION_MODE = 'stress'  # debug | stress | real
PHYSIONET_USERNAME = ''  # fill this, e.g. 'pidoxy'

WORK_DIR = '/kaggle/working/physionet'
PILOT_OUTPUT_DIR = '/kaggle/working/rgca_pilot_500'
SUITE_OUTPUT_DIR = '/kaggle/working/rgca_experiments/mimic_pilot_baseline_v0'

RETRIEVAL_LIMIT = 400
EVAL_LIMIT = 100

print('EXECUTION_MODE:', EXECUTION_MODE)
print('PHYSIONET_USERNAME set:', bool(PHYSIONET_USERNAME))


## 3. Prepare Dataset From PhysioNet

Run this if `/kaggle/working/rgca_pilot_500/data/mimic_subset.jsonl` does not exist.

This downloads only:

- `mimic-cxr-2.0.0-metadata.csv.gz`
- `mimic-cxr-2.0.0-split.csv.gz`
- `mimic-cxr-2.0.0-chexpert.csv.gz`
- `mimic-cxr-reports.zip`

It does not download the full image dataset. That is okay for `stress` mode. `real` mode requires images later.


In [ ]:
%cd /kaggle/working/RGCA

from pathlib import Path
subset = Path(PILOT_OUTPUT_DIR) / 'data' / 'mimic_subset.jsonl'

if subset.exists():
    print('Subset already exists, skipping preparation:', subset)
else:
    if not PHYSIONET_USERNAME:
        raise ValueError('Set PHYSIONET_USERNAME before preparing the dataset.')
    !python scripts/kaggle_prepare_mimic_subset.py       --physionet-user {PHYSIONET_USERNAME}       --work-dir {WORK_DIR}       --output-dir {PILOT_OUTPUT_DIR}       --retrieval-limit {RETRIEVAL_LIMIT}       --eval-limit {EVAL_LIMIT}


## 4. Run Guarded Baseline Suite

This command validates the subset, writes fingerprints/manifests, runs the suite, summarizes tables, and packages a private dataset zip.


In [ ]:
%cd /kaggle/working/RGCA

!python scripts/kaggle_bootstrap_baseline.py   --subset-jsonl {PILOT_OUTPUT_DIR}/data/mimic_subset.jsonl   --execution-mode {EXECUTION_MODE}   --output-dir {SUITE_OUTPUT_DIR}   --pilot-output-dir {PILOT_OUTPUT_DIR}   --overwrite


## 5. Inspect Results


In [ ]:
from pathlib import Path

summary = Path(SUITE_OUTPUT_DIR) / 'bootstrap_summary.json'
table = Path(SUITE_OUTPUT_DIR) / 'tables' / 'suite_summary.md'
manifest = Path(SUITE_OUTPUT_DIR) / 'suite_manifest.json'

for path in [summary, manifest, table]:
    print(path, 'exists=', path.exists())

if table.exists():
    print(table.read_text(encoding='utf-8')[:6000])


## 6. Save Private Dataset

After the suite succeeds, save this zip as a private Kaggle dataset:

```text
/kaggle/working/rgca_private_dataset.zip
```

In Kaggle:

1. Open the Output panel.
2. Download or locate `rgca_private_dataset.zip`.
3. Create a new Kaggle Dataset.
4. Upload the zip contents.
5. Set visibility to **Private**.
6. Future notebooks can attach it as input and skip dataset preparation.
